In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

def swish(x):
    return tf.nn.swish(x)

def conv_block(x, filters, kernel=3, strides=1, se_ratio=0.0, drop_rate=0.0):
    # Depthwise separable conv block with BN + swish and optional SE
    x = layers.SeparableConv2D(filters, kernel, padding='same', strides=strides, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)
    if se_ratio and se_ratio > 0:
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Reshape((1,1,filters))(se)
        se = layers.Conv2D(max(1, int(filters*se_ratio)), 1, activation='swish', padding='same')(se)
        se = layers.Conv2D(filters, 1, activation='sigmoid', padding='same')(se)
        x = layers.Multiply()([x, se])
    if drop_rate and drop_rate > 0:
        x = layers.Dropout(drop_rate)(x)
    return x

def residual_down(x, filters, stride, se_ratio=0.0):
    shortcut = layers.SeparableConv2D(filters, 1, strides=stride, padding='same', use_bias=False)(x)
    shortcut = layers.BatchNormalization()(shortcut)
    x = conv_block(x, filters, strides=stride, se_ratio=se_ratio)
    x = conv_block(x, filters, strides=1, se_ratio=se_ratio)
    x = layers.Add()([x, shortcut])
    return x

def build_compact_cnn(input_shape=(224,224,3), num_classes=1, filters_multiplier=1.0, dropout_head=0.2, se_ratio=0.0):
    # filters_multiplier lets you scale model up/down. 1.0 ~ default small model.
    def F(n): return max(8, int(n * filters_multiplier))
    inp = layers.Input(shape=input_shape)

    # Stem
    x = layers.Conv2D(F(16), 3, strides=2, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)

    # Stages: (filters, repeats, stride)
    stages = [
        (24, 1, 1),
        (40, 2, 2),
        (80, 3, 2),
        (160, 3, 2),
    ]
    for filters, repeats, stride in stages:
        for i in range(repeats):
            s = stride if i == 0 else 1
            x = residual_down(x, F(filters), stride=s, se_ratio=se_ratio)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(F(128), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)
    x = layers.Dropout(dropout_head)(x)
    out = layers.Dense(num_classes, activation='sigmoid')(x)

    model = models.Model(inputs=inp, outputs=out)
    return model

# Exemple : construire le modèle
model = build_compact_cnn(input_shape=(224,224,3), filters_multiplier=1.0, dropout_head=0.3, se_ratio=0.125)
model.summary()
